# 05 · Gene perturbation prediction (GEARS + HeteroSC)

Routing unit = **gene**. Each foundation model supplies a gene-embedding table; HeteroSC fuses them and the fused matrix replaces GEARS' gene-embedding layer.
Two ways to use it:

* **End-to-end fusion** (Fusion Ladder, adaptive-vs-uniform ablation): adapters + router are trained through the GEARS objective; every other GEARS parameter is frozen.
* **Initialisation**: after training, export the fused matrix and give it to a fresh GEARS model (frozen or unfrozen).

> **Say which one you used.** If the fusion module is *not* trained before export, the fused matrix is a fixed random projection of the foundation-model embeddings.

Needs: GEARS (`pip install cell-gears` plus PyTorch Geometric), `data/gene_perturbation/` (notebook 01) and gene-embedding tables from the four foundation models.

In [ ]:
from pathlib import Path
import numpy as np, torch

from gears import PertData, GEARS
from heterosc.perturbation import (align_gene_table, attach_heterosc_to_gears, export_fused_matrix, expert_combinations)

DATA = Path("data/gene_perturbation")
DATASET = "norman"                        # or "adamson"
SPLIT_SEED, TRAIN_SET_SIZE = 3, 0.75      # split provided in the Zenodo record (norman_simulation_3_0.75.pkl)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

pert = PertData(str(DATA))
pert.load(data_path=str(DATA / DATASET))
pert.prepare_split(split="simulation", seed=SPLIT_SEED, train_gene_set_size=TRAIN_SET_SIZE)
pert.get_dataloader(batch_size=8, test_batch_size=32)
genes = pert.adata.var["gene_name"].tolist()
print(len(genes), "genes")

## Gene-embedding tables, aligned **by gene identity**
Each foundation model has its own gene vocabulary. `align_gene_table` maps GEARS genes to each model's table by name (genes the model does not know get N(0,1) vectors,
GEARS' own default) and reports the coverage. **Do not fill tables by row position**: that gives arbitrary vectors.

In [ ]:
from heterosc.extraction.geneformer import geneformer_gene_table, symbol_to_ensembl
from heterosc.extraction.scgpt import scgpt_gene_table
from heterosc.extraction.scfoundation import scfoundation_gene_table
# CellFM: build the table inside its environment with heterosc.extraction.cellfm.cellfm_gene_table(net, gene_names)
# (gene_names must be CellFM's vocabulary in row order) and save it as {gene: vector}, then load it here.

tables = {}
gf = geneformer_gene_table("checkpoints/Geneformer");   tables["geneformer"]   = align_gene_table(genes, gf, 768, key_fn=symbol_to_ensembl("checkpoints/Geneformer"))
sg = scgpt_gene_table("checkpoints/scGPT");             tables["scgpt"]        = align_gene_table(genes, sg, 512)
sf = scfoundation_gene_table("checkpoints/scFoundation", "checkpoints/scFoundation/OS_scRNA_gene_index.19264.tsv"); tables["scfoundation"] = align_gene_table(genes, sf, 768)
# tables["cellfm"] = align_gene_table(genes, cellfm_table, 1536)

for k, (mat, found) in tables.items():
    print(f"{k:<13} vocabulary coverage {found.mean():.1%}")
expert_tables = {k: mat for k, (mat, _) in tables.items()}

## Stage 1 · GEARS without pretrained gene embeddings
The reference model that stage 2 starts from (`use_pretrained=False`).

In [ ]:
HIDDEN = 512
g1 = GEARS(pert, device=DEVICE)
g1.model_initialize(hidden_size=HIDDEN, use_pretrained=False)
g1.train(epochs=15, lr=0.005)
stage1_state = {k: v.cpu() for k, v in g1.model.state_dict().items()}

## Stage 2 · replace the gene-embedding layer by HeteroSC and train adapters + router only
`fusion="adaptive"` is HeteroSC; `fusion="uniform"` is the equal-weight control (α = 1/K) with identical adapters.

In [ ]:
def stage2(fusion, names):
    g = GEARS(pert, device=DEVICE)
    g.model_initialize(hidden_size=HIDDEN, use_pretrained=False)
    g.model.load_state_dict(stage1_state)
    module = attach_heterosc_to_gears(g, {n: expert_tables[n] for n in names}, d_shared=HIDDEN, fusion=fusion, device=DEVICE)
    print(f"{fusion} / {names}: trainable parameters = {module.num_trainable_params():,}")
    g.train(epochs=5, lr=0.005)
    return g, module

g_adaptive, mod_adaptive = stage2("adaptive", list(expert_tables))
g_uniform,  mod_uniform  = stage2("uniform",  list(expert_tables))

In [ ]:
fused, alpha = export_fused_matrix(mod_adaptive)               # (n_genes, 512) and per-gene routing weights
print("mean routing weight per expert:", dict(zip(expert_tables, alpha.mean(0).numpy().round(3))))
# Evaluate with GEARS' own tools (Pearson / MSE on the top-20 DE genes: Pearson DE, MSE DE):
from gears.inference import evaluate, compute_metrics
test_res = evaluate(pert.dataloader["test_loader"], g_adaptive.best_model, False, DEVICE)
metrics, _ = compute_metrics(test_res); metrics

## Fusion Ladder: all 15 expert combinations
Repeat stage 2 for every non-empty subset of the experts (singles → full four-expert fusion) to see whether performance grows as experts are added.

In [ ]:
combos = expert_combinations(list(expert_tables)); print(len(combos), "combinations")
RUN_LADDER = False
if RUN_LADDER:
    ladder = {}
    for combo in combos:
        g, m = stage2("adaptive", list(combo))
        r, _ = compute_metrics(evaluate(pert.dataloader["test_loader"], g.best_model, False, DEVICE))
        ladder[combo] = r["pearson_de"]

**GEARS compatibility notes.** (1) With some pandas versions GEARS fails on `self.adata.X[self.adata.obs.condition == 'ctrl']`; converting the mask with
`.to_numpy()` in GEARS' source fixes it. (2) GEARS' constructor and `train` signatures differ between versions; check them before running.